# Oracle Agentic Audit — offline research replay

This notebook replays the checksum-indexed **real research bundle** without an LLM or GPU. It does not download an unpinned repository. Hosted Colab status: **NOT RUN** until a real fresh hosted execution and its outputs are recorded. The synthetic smoke check is separate. The bundle uses a file-hash snapshot; a public repository commit and data revision remain release requirements.


In [ ]:
import sys, pathlib, tarfile, subprocess, json
assert sys.version_info[:2] == (3, 12), 'This lock was validated on Python 3.12; record other runtimes as unverified.'
from google.colab import files
uploaded = files.upload()  # Select oracle-camera-ready-20261009.tar.gz
archives = [pathlib.Path(name) for name in uploaded if name.endswith('.tar.gz')]
assert len(archives) == 1, 'Provide exactly one governed research bundle.'
with tarfile.open(archives[0]) as archive:
    archive.extractall('/content/oracle-replay', filter='data')
root = pathlib.Path('/content/oracle-replay/Oracle-Agentic-Audit')
assert (root/'bundle_manifest.json').is_file()


Create an isolated environment; the notebook kernel itself is not modified. The CPU replay includes fixed LightGBM training, calibration, 2,000-case bootstraps, lineage audits and 100 shuffled-label controls. It can take several minutes.


In [ ]:
environment = pathlib.Path('/content/oracle-replay-env')
subprocess.run([sys.executable, '-m', 'venv', str(environment)], check=True)
python = str(environment/'bin/python')
subprocess.run([python, '-m', 'pip', 'install', '-r', str(root/'requirements.lock')], check=True)
subprocess.run([python, '-m', 'pip', 'check'], check=True)
subprocess.run([python, '-m', 'experiments.verify_bundle'], cwd=root, check=True)
subprocess.run([python, '-m', 'oracle_audit', 'experiment-check'], cwd=root, check=True)


In [ ]:
subprocess.run([python, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=root, check=True)
subprocess.run([python, '-m', 'analysis.replay', '--retrain', '--out', 'outputs/colab_replay'], cwd=root, check=True)
subprocess.run([python, '-m', 'analysis.verify_replay', '--reference', 'outputs/final', '--candidate', 'outputs/colab_replay'], cwd=root, check=True)


In [ ]:
from IPython.display import display, Markdown, Image
display(Markdown((root/'outputs/colab_replay/RESULTS_zh.md').read_text()))
for name in ['calibration','citation_errors','cost_performance','guard_ablation']:
    path = root/'outputs/colab_replay/figures'/f'{name}.png'
    if path.exists(): display(Image(filename=str(path)))


Only the following cell, executed in an actual hosted Colab runtime after every check passes, records hosted success. Save the notebook with its executed outputs and download the receipt. A locally generated or unexecuted cell is not a hosted receipt.


In [ ]:
import datetime, platform, os
assert os.environ.get('COLAB_RELEASE_TAG') or os.environ.get('COLAB_BACKEND_VERSION'), 'Hosted Colab environment was not identified.'
comparison=json.loads((root/'outputs/colab_replay/comparison_receipt.json').read_text())
assert comparison['status']=='PASS'
receipt={'status':'PASS','environment':'actual hosted Google Colab CPU','utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'python':platform.python_version(),'platform':platform.platform(),'tables':comparison['compared_tables']}
path=root/'manifests/hosted_colab_receipt.json'
path.write_text(json.dumps(receipt,indent=2)+'\n')
files.download(str(path))
